📚 **Importación de libreías**

In [ ]:
from pathlib import Path
import sys

# Pandas: Para manipulación y análisis de datos tabulares.
import pandas as pd

# NumPy: Para operaciones matemáticas y manejo de arrays.
import numpy as np

# Scikit-learn: Para preprocesamiento de datos, selección de características y transformación.
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2

# Feature-engine: Para técnicas avanzadas de ingeniería de características.
from feature_engine.imputation import MeanMedianImputer
from feature_engine.encoding import OneHotEncoder

💾 **Carga de datos**

In [2]:
# Definir rutas de datos
data_path = Path("../../data")
bronze_path = data_path / "Bronze"

# Verificar que la carpeta existe
if not bronze_path.exists():
    print(f"❌ No se encontró la carpeta: {bronze_path}")
    sys.exit(1)

# Cargar el dataset principal
dataset_file = bronze_path / "Dataset.csv"

if dataset_file.exists():
    print(f"📂 Cargando datos desde: {dataset_file}")
    df = pd.read_csv(dataset_file)

    # Información básica del dataset
    print(f"✅ Dataset cargado exitosamente")
    print(f"📊 Forma del dataset: {df.shape}")
    print(f"📋 Columnas: {list(df.columns)}")
    print(f"💾 Memoria utilizada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


📂 Cargando datos desde: ../../data/Bronze/Dataset.csv
✅ Dataset cargado exitosamente
📊 Forma del dataset: (1744667, 27)
📋 Columnas: ['CreditoID', 'PersonaID', 'DepartamentoResidencia', 'DepartamentoMayorFrecuenciaCompra', 'AlmacenMayorFrecuenciaPago', 'ValorPagosUltimosMes', 'AlmacenCredito', 'RiesgoAlmacen', 'DepartamentoCredito', 'FechaCredito', 'ValorCredito', 'CupoTotal', 'CupoDisponibleTotal', 'storeIdEventoA', 'FechaEventoA', 'EventoA', 'FechaEventoB', 'LocalizacionEventoB', 'LocalizacionComercioCredito', 'StatusComercioCredito', 'FrecuenciaCreditosSemana', 'CantidadCreditosUltimaSemana', 'ValorAtipicoCliente', 'ValorAtipicoComercio', 'TipoAlmacenCredito', 'TipoCliente', 'Atipico']
💾 Memoria utilizada: 1540.94 MB


**1. Limpieza de datos**
En esta etapa inicial, se realizaron tareas esenciales para asegurar la calidad y coherencia de los datos:

- Eliminación de duplicados
- Tratamiento de valores faltantes
- Atípicos

In [3]:
# Código de conversión
# Crear una copia para trabajar
df_converted = df.copy()

# 🆔 VARIABLES CATEGÓRICAS NOMINALES (IDs)
categorical_ids = [
    "CreditoID",
    "PersonaID",
    "AlmacenMayorFrecuenciaPago",
    "AlmacenCredito",
    "storeIdEventoA",
]

for col in categorical_ids:
    if col in df_converted.columns:
        df_converted[col] = df_converted[col].astype("string")
        print(f"   ✅ {col}: Convertido a string (ID categórico)")

# 🏷️ VARIABLES CATEGÓRICAS NOMINALES (Códigos/Regiones)
categorical_nominal = [
    "DepartamentoResidencia",
    "DepartamentoMayorFrecuenciaCompra",
    "DepartamentoCredito",
    "TipoAlmacenCredito",
]

for col in categorical_nominal:
    if col in df_converted.columns:
        df_converted[col] = df_converted[col].astype("category")
        print(f"   ✅ {col}: Convertido a category (nominal)")

# 📊 VARIABLES CATEGÓRICAS ORDINALES
categorical_ordinal = ["RiesgoAlmacen", "StatusComercioCredito", "TipoCliente"]

for col in categorical_ordinal:
    if col in df_converted.columns:
        df_converted[col] = df_converted[col].astype("category")
        print(f"   ✅ {col}: Convertido a category (ordinal)")

# 📅 VARIABLES DE FECHA
date_columns = ["FechaCredito", "FechaEventoA", "FechaEventoB"]

for col in date_columns:
    if col in df_converted.columns:
        try:
            # Intentar múltiples formatos de fecha
            df_converted[col] = pd.to_datetime(
                df_converted[col], errors="coerce", infer_datetime_format=True
            )
            print(f"   ✅ {col}: Convertido a datetime")
        except:
            print(f"   ❌ {col}: Error al convertir a datetime")

# 💰 VARIABLES NUMÉRICAS CONTINUAS
numeric_continuous = [
    "ValorPagosUltimosMes",
    "ValorCredito",
    "CupoTotal",
    "CupoDisponibleTotal",
    "ValorAtipicoCliente",
    "ValorAtipicoComercio",
]

for col in numeric_continuous:
    if col in df_converted.columns:
        try:
            df_converted[col] = pd.to_numeric(df_converted[col], errors="coerce")
            print(f"   ✅ {col}: Convertido a numeric (continua)")
        except:
            print(f"   ❌ {col}: Error al convertir a numeric")

# 🔢 VARIABLES NUMÉRICAS DISCRETAS (Conteos)
numeric_discrete = ["FrecuenciaCreditosSemana", "CantidadCreditosUltimaSemana"]

for col in numeric_discrete:
    if col in df_converted.columns:
        try:
            df_converted[col] = pd.to_numeric(
                df_converted[col], errors="coerce"
            ).astype("Int64")
            print(f"   ✅ {col}: Convertido a Int64 (discreta)")
        except:
            print(f"   ❌ {col}: Error al convertir a Int64")

# ✅ VARIABLES BINARIAS
# EventoA: 'SI' vs NaN -> 1/0
if "EventoA" in df_converted.columns:
    df_converted["EventoA"] = (df_converted["EventoA"] == "SI").astype("Int64")
    print(f"   ✅ EventoA: Convertido a binario (1=SI, 0=NaN)")

# Atipico: Variable objetivo binaria
if "Atipico" in df_converted.columns:
    df_converted["Atipico"] = df_converted["Atipico"].astype("Int64")
    print(f"   ✅ Atipico: Convertido a Int64 (target binario)")

# 🗺️ VARIABLES GEOESPACIALES
geo_columns = ["LocalizacionEventoB", "LocalizacionComercioCredito"]

for col in geo_columns:
    if col in df_converted.columns:
        print(
            f"   📍 {col}: Mantenido como string (requiere procesamiento geoespacial)"
        )

print(f"\\n📊 RESUMEN DE CONVERSIONES APLICADAS")
print("=" * 70)

# Comparar tipos antes y después
changes_made = []
for col in df.columns:
    if col in df_converted.columns:
        old_type = str(df[col].dtype)
        new_type = str(df_converted[col].dtype)
        if old_type != new_type:
            changes_made.append(
                {"Columna": col, "Tipo_Original": old_type, "Tipo_Nuevo": new_type}
            )

if changes_made:
    changes_df = pd.DataFrame(changes_made)
    display(changes_df)
else:
    print("No se realizaron cambios de tipo de datos")

# Actualizar el dataframe principal
print(f"\\n✅ Dataset actualizado con nuevos tipos de datos")


   ✅ CreditoID: Convertido a string (ID categórico)
   ✅ PersonaID: Convertido a string (ID categórico)
   ✅ AlmacenMayorFrecuenciaPago: Convertido a string (ID categórico)
   ✅ AlmacenCredito: Convertido a string (ID categórico)
   ✅ storeIdEventoA: Convertido a string (ID categórico)
   ✅ DepartamentoResidencia: Convertido a category (nominal)
   ✅ DepartamentoMayorFrecuenciaCompra: Convertido a category (nominal)
   ✅ DepartamentoCredito: Convertido a category (nominal)
   ✅ TipoAlmacenCredito: Convertido a category (nominal)
   ✅ RiesgoAlmacen: Convertido a category (ordinal)
   ✅ StatusComercioCredito: Convertido a category (ordinal)
   ✅ TipoCliente: Convertido a category (ordinal)


/tmp/ipykernel_13304/4278647523.py:47: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_converted[col] = pd.to_datetime(
/tmp/ipykernel_13304/4278647523.py:47: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_converted[col] = pd.to_datetime(
/tmp/ipykernel_13304/4278647523.py:47: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_converted[col

   ✅ FechaCredito: Convertido a datetime
   ✅ FechaEventoA: Convertido a datetime
   ✅ FechaEventoB: Convertido a datetime
   ✅ ValorPagosUltimosMes: Convertido a numeric (continua)
   ✅ ValorCredito: Convertido a numeric (continua)
   ✅ CupoTotal: Convertido a numeric (continua)
   ✅ CupoDisponibleTotal: Convertido a numeric (continua)
   ✅ ValorAtipicoCliente: Convertido a numeric (continua)
   ✅ ValorAtipicoComercio: Convertido a numeric (continua)
   ✅ FrecuenciaCreditosSemana: Convertido a Int64 (discreta)
   ✅ CantidadCreditosUltimaSemana: Convertido a Int64 (discreta)
   ✅ EventoA: Convertido a binario (1=SI, 0=NaN)
   ✅ Atipico: Convertido a Int64 (target binario)
   📍 LocalizacionEventoB: Mantenido como string (requiere procesamiento geoespacial)
   📍 LocalizacionComercioCredito: Mantenido como string (requiere procesamiento geoespacial)
\n📊 RESUMEN DE CONVERSIONES APLICADAS


,Columna,Tipo_Original,Tipo_Nuevo
0,CreditoID,object,string
1,PersonaID,object,string
2,DepartamentoResidencia,int64,category
3,DepartamentoMayorFrecuenciaCompra,int64,category
4,AlmacenMayorFrecuenciaPago,object,string
5,AlmacenCredito,object,string
6,RiesgoAlmacen,int64,category
7,DepartamentoCredito,int64,category
8,FechaCredito,object,"datetime64[ns, UTC]"
9,storeIdEventoA,object,string


\n✅ Dataset actualizado con nuevos tipos de datos


In [ ]:
# Verificar y eliminar duplicados por CreditoID
print("🔍 Análisis de duplicados por CreditoID")
print("=" * 50)

# Contar registros duplicados por CreditoID
duplicados = df_converted["CreditoID"].duplicated().sum()
print(f"📊 Registros duplicados encontrados: {duplicados}")
print(f"📊 Total de registros: {len(df_converted)}")

if duplicados > 0:
    # Mostrar algunos ejemplos de CreditoID duplicados
    creditos_duplicados = (
        df_converted[df_converted["CreditoID"].duplicated(keep=False)]["CreditoID"]
        .value_counts()
        .head()
    )
    print(f"\n🔍 Ejemplos de CreditoID duplicados:")
    print(creditos_duplicados)

    # Eliminar duplicados manteniendo el primer registro
    df_converted = df_converted.drop_duplicates(subset=["CreditoID"], keep="first")
    print(f"\n✅ Duplicados eliminados. Registros restantes: {len(df_converted)}")
else:
    print("✅ No se encontraron duplicados por CreditoID")


🔍 Análisis de duplicados por CreditoID
📊 Registros duplicados encontrados: 0
📊 Total de registros: 1744667
✅ No se encontraron duplicados por CreditoID


In [5]:
# 🔍 IDENTIFICACIÓN Y CORRECCIÓN DE INCONSISTENCIAS EN VARIABLES NUMÉRICAS
# (SOLO PARA REGISTROS NO ATÍPICOS)
print("🔍 ANÁLISIS DE INCONSISTENCIAS EN VARIABLES NUMÉRICAS")
print("=" * 60)
print("⚠️  NOTA: Solo se corrigen registros con Atipico == 0")
print("🔒 Los registros con Atipico == 1 se mantienen intactos")
print("=" * 60)

# Crear copia para trabajar
df_clean = df_converted.copy()

# Crear máscara para registros NO atípicos (donde podemos hacer correcciones)
mask_no_atipicos = df_clean["Atipico"] == 0
registros_no_atipicos = mask_no_atipicos.sum()
registros_atipicos = (df_clean["Atipico"] == 1).sum()

print(f"📊 Total registros: {len(df_clean):,}")
print(f"📊 Registros NO atípicos (corregibles): {registros_no_atipicos:,}")
print(f"📊 Registros atípicos (protegidos): {registros_atipicos:,}")

# 1. CUPO DISPONIBLE NEGATIVO (INCONSISTENCIA CRÍTICA)
print("\n1️⃣ CUPO DISPONIBLE NEGATIVO")
print("-" * 40)
negativos_cupo_total = (df_clean["CupoDisponibleTotal"] < 0).sum()
negativos_cupo_no_atipicos = (
    mask_no_atipicos & (df_clean["CupoDisponibleTotal"] < 0)
).sum()

print(f"📊 Total registros con cupo negativo: {negativos_cupo_total:,}")
print(f"📊 En registros NO atípicos: {negativos_cupo_no_atipicos:,}")
print(f"📊 Valor mínimo actual: {df_clean['CupoDisponibleTotal'].min():,.2f}")

if negativos_cupo_no_atipicos > 0:
    df_clean.loc[
        mask_no_atipicos & (df_clean["CupoDisponibleTotal"] < 0), "CupoDisponibleTotal"
    ] = np.nan
    print(
        f"✅ Convertidos {negativos_cupo_no_atipicos:,} valores negativos a NaN (solo NO atípicos)"
    )

# 2. CUPO TOTAL = 0 PERO CON CUPO DISPONIBLE > 0
print("\n2️⃣ CUPO TOTAL CERO CON CUPO DISPONIBLE POSITIVO")
print("-" * 40)
inconsistencia_cupo_total = (
    (df_clean["CupoTotal"] == 0) & (df_clean["CupoDisponibleTotal"] > 0)
).sum()
inconsistencia_cupo_no_atipicos = (
    mask_no_atipicos
    & (df_clean["CupoTotal"] == 0)
    & (df_clean["CupoDisponibleTotal"] > 0)
).sum()

print(f"📊 Total registros inconsistentes: {inconsistencia_cupo_total:,}")
print(f"📊 En registros NO atípicos: {inconsistencia_cupo_no_atipicos:,}")

if inconsistencia_cupo_no_atipicos > 0:
    mask_inconsistencia = (
        mask_no_atipicos
        & (df_clean["CupoTotal"] == 0)
        & (df_clean["CupoDisponibleTotal"] > 0)
    )
    df_clean.loc[mask_inconsistencia, "CupoDisponibleTotal"] = np.nan
    print(
        f"✅ Corregidos {inconsistencia_cupo_no_atipicos:,} casos inconsistentes (solo NO atípicos)"
    )

# 3. VALOR CRÉDITO > CUPO TOTAL (cuando cupo > 0)
print("\n3️⃣ VALOR CRÉDITO MAYOR AL CUPO TOTAL")
print("-" * 40)
credito_mayor_cupo_total = (
    (df_clean["ValorCredito"] > df_clean["CupoTotal"]) & (df_clean["CupoTotal"] > 0)
).sum()
credito_mayor_cupo_no_atipicos = (
    mask_no_atipicos
    & (df_clean["ValorCredito"] > df_clean["CupoTotal"])
    & (df_clean["CupoTotal"] > 0)
).sum()

print(f"📊 Total donde crédito > cupo total: {credito_mayor_cupo_total:,}")
print(f"📊 En registros NO atípicos: {credito_mayor_cupo_no_atipicos:,}")

if credito_mayor_cupo_no_atipicos > 0:
    mask_credito_mayor = (
        mask_no_atipicos
        & (df_clean["ValorCredito"] > df_clean["CupoTotal"])
        & (df_clean["CupoTotal"] > 0)
    )
    df_clean.loc[mask_credito_mayor, "ValorCredito"] = np.nan
    print(
        f"✅ Convertidos {credito_mayor_cupo_no_atipicos:,} valores inconsistentes a NaN (solo NO atípicos)"
    )

# 4. VALORES EXTREMADAMENTE ALTOS (OUTLIERS EXTREMOS)
print("\n4️⃣ VALORES EXTREMADAMENTE ALTOS (OUTLIERS)")
print("-" * 40)

variables_numericas = [
    "ValorPagosUltimosMes",
    "ValorCredito",
    "CupoTotal",
    "ValorAtipicoCliente",
    "ValorAtipicoComercio",
]

outliers_corregidos = 0
for var in variables_numericas:
    if var in df_clean.columns:
        # Calcular umbrales usando solo registros NO atípicos
        data_no_atipicos = df_clean.loc[mask_no_atipicos, var]
        Q1 = data_no_atipicos.quantile(0.25)
        Q3 = data_no_atipicos.quantile(0.75)
        IQR = Q3 - Q1
        umbral_superior = Q3 + 3 * IQR

        outliers_total = (df_clean[var] > umbral_superior).sum()
        outliers_no_atipicos = (
            mask_no_atipicos & (df_clean[var] > umbral_superior)
        ).sum()

        print(
            f"   {var}: Total outliers {outliers_total:,} | NO atípicos {outliers_no_atipicos:,} | Umbral: {umbral_superior:,.0f}"
        )

        if outliers_no_atipicos > 0:
            df_clean.loc[mask_no_atipicos & (df_clean[var] > umbral_superior), var] = (
                np.nan
            )
            outliers_corregidos += outliers_no_atipicos

print(
    f"✅ Total outliers extremos convertidos a NaN: {outliers_corregidos:,} (solo NO atípicos)"
)

# 5. FRECUENCIAS IMPOSIBLES
print("\n5️⃣ FRECUENCIAS IMPOSIBLES")
print("-" * 40)

# Frecuencia > 7 créditos por semana (más de 1 por día)
freq_alta_total = (df_clean["FrecuenciaCreditosSemana"] > 7).sum()
freq_alta_no_atipicos = (
    mask_no_atipicos & (df_clean["FrecuenciaCreditosSemana"] > 7)
).sum()

print(
    f"📊 Frecuencia > 7 créditos/semana: Total {freq_alta_total:,} | NO atípicos {freq_alta_no_atipicos:,}"
)

if freq_alta_no_atipicos > 0:
    df_clean.loc[
        mask_no_atipicos & (df_clean["FrecuenciaCreditosSemana"] > 7),
        "FrecuenciaCreditosSemana",
    ] = np.nan
    print(
        f"✅ Convertidos {freq_alta_no_atipicos:,} valores imposibles a NaN (solo NO atípicos)"
    )

# Cantidad créditos última semana > 10 (muy improbable)
cant_alta_total = (df_clean["CantidadCreditosUltimaSemana"] > 10).sum()
cant_alta_no_atipicos = (
    mask_no_atipicos & (df_clean["CantidadCreditosUltimaSemana"] > 10)
).sum()

print(
    f"📊 Cantidad > 10 créditos última semana: Total {cant_alta_total:,} | NO atípicos {cant_alta_no_atipicos:,}"
)

if cant_alta_no_atipicos > 0:
    df_clean.loc[
        mask_no_atipicos & (df_clean["CantidadCreditosUltimaSemana"] > 10),
        "CantidadCreditosUltimaSemana",
    ] = np.nan
    print(
        f"✅ Convertidos {cant_alta_no_atipicos:,} valores improbables a NaN (solo NO atípicos)"
    )

# 6. RESUMEN FINAL
print(f"\n📊 RESUMEN DE CORRECCIONES APLICADAS")
print("=" * 60)

# Verificar que los registros atípicos no fueron modificados
registros_atipicos_final = (df_clean["Atipico"] == 1).sum()
print(
    f"🔒 Registros atípicos preservados: {registros_atipicos_final:,} (deben ser {registros_atipicos:,})"
)

if registros_atipicos_final == registros_atipicos:
    print("✅ CONFIRMADO: Ningún registro atípico fue modificado")
else:
    print("❌ ERROR: Se modificaron registros atípicos")

# Comparar antes y después solo para registros NO atípicos
variables_numericas_todas = df_clean.select_dtypes(include=[np.number]).columns.tolist()

for var in variables_numericas_todas:
    if var in df_converted.columns and var != "Atipico":
        nans_antes = df_converted.loc[mask_no_atipicos, var].isnull().sum()
        nans_despues = df_clean.loc[mask_no_atipicos, var].isnull().sum()
        diferencia = nans_despues - nans_antes

        if diferencia > 0:
            print(f"   {var}: +{diferencia:,} NaN en registros NO atípicos")

# Actualizar el dataframe principal
df_converted = df_clean.copy()
print(f"\n✅ Dataset actualizado con correcciones aplicadas")
print(f"📊 Shape final: {df_converted.shape}")
print(f"🔒 Registros atípicos (Atipico==1) permanecen intactos")


🔍 ANÁLISIS DE INCONSISTENCIAS EN VARIABLES NUMÉRICAS
⚠️  NOTA: Solo se corrigen registros con Atipico == 0
🔒 Los registros con Atipico == 1 se mantienen intactos
📊 Total registros: 1,744,667
📊 Registros NO atípicos (corregibles): 1,744,264
📊 Registros atípicos (protegidos): 403

1️⃣ CUPO DISPONIBLE NEGATIVO
----------------------------------------
📊 Total registros con cupo negativo: 1,949
📊 En registros NO atípicos: 1,835
📊 Valor mínimo actual: -4,278,013.00
✅ Convertidos 1,835 valores negativos a NaN (solo NO atípicos)

2️⃣ CUPO TOTAL CERO CON CUPO DISPONIBLE POSITIVO
----------------------------------------
📊 Total registros inconsistentes: 1,293
📊 En registros NO atípicos: 1,042
✅ Corregidos 1,042 casos inconsistentes (solo NO atípicos)

3️⃣ VALOR CRÉDITO MAYOR AL CUPO TOTAL
----------------------------------------
📊 Total donde crédito > cupo total: 298
📊 En registros NO atípicos: 292
✅ Convertidos 292 valores inconsistentes a NaN (solo NO atípicos)

4️⃣ VALORES EXTREMADAMENTE ALT

In [6]:
print(f"\n❌ VALORES FALTANTES")
print("=" * 50)
missing_data = df_converted.isnull().sum()
missing_percent = (missing_data / len(df_converted)) * 100
missing_df = pd.DataFrame(
    {
        "Columna": missing_data.index,
        "Valores_Faltantes": missing_data.values,
        "Porcentaje": missing_percent.values,
    }
).sort_values("Porcentaje", ascending=False)

display(missing_df[missing_df["Valores_Faltantes"] > 0])



❌ VALORES FALTANTES


,Columna,Valores_Faltantes,Porcentaje
14,FechaEventoA,1657617,95.010509
13,storeIdEventoA,1657617,95.010509
17,LocalizacionEventoB,1628802,93.358905
16,FechaEventoB,1628802,93.358905
23,ValorAtipicoComercio,112954,6.474244
10,ValorCredito,66031,3.784734
22,ValorAtipicoCliente,46121,2.643542
5,ValorPagosUltimosMes,28114,1.611425
21,CantidadCreditosUltimaSemana,24849,1.424283
12,CupoDisponibleTotal,2877,0.164903


In [ ]:
# 🔧 PROCESO REAL DE LIMPIEZA DE DATOS
print("🔧 PROCESO REAL DE LIMPIEZA DE DATOS")
print("=" * 60)

# 1. ELIMINAR COLUMNAS ESPECÍFICAS
print("\n1️⃣ ELIMINANDO COLUMNAS ESPECÍFICAS")
print("-" * 40)

columnas_a_eliminar = [
    "CreditoID",
    "PersonaID",
    "storeIdEventoA",
    "FechaEventoA",
    "EventoA",
    "FechaEventoB",
    "LocalizacionEventoB",
    "LocalizacionComercioCredito",
]

# Verificar cuáles columnas existen antes de eliminar
columnas_existentes = [
    col for col in columnas_a_eliminar if col in df_converted.columns
]
columnas_no_encontradas = [
    col for col in columnas_a_eliminar if col not in df_converted.columns
]

print(f"📊 Columnas a eliminar encontradas: {len(columnas_existentes)}")
for col in columnas_existentes:
    print(f"   ✅ {col}")

if columnas_no_encontradas:
    print(f"\n⚠️ Columnas no encontradas: {len(columnas_no_encontradas)}")
    for col in columnas_no_encontradas:
        print(f"   ❌ {col}")

# Eliminar columnas existentes
df_limpio = df_converted.drop(columns=columnas_existentes)

print(f"\n📊 Columnas antes: {df_converted.shape[1]}")
print(f"📊 Columnas después: {df_limpio.shape[1]}")
print(f"📊 Columnas eliminadas: {len(columnas_existentes)}")

# 2. ELIMINAR REGISTROS CON NaN (SOLO ATÍPICO == 0)
print("\n2️⃣ ELIMINANDO REGISTROS CON NaN (SOLO ATÍPICO == 0)")
print("-" * 40)

# Estado inicial
print(f"📊 ESTADO INICIAL:")
print(f"   Total registros: {len(df_limpio):,}")
print(f"   Registros atípicos (1): {(df_limpio['Atipico'] == 1).sum():,}")
print(f"   Registros normales (0): {(df_limpio['Atipico'] == 0).sum():,}")

# Identificar registros a eliminar: tienen NaN Y son atípico == 0
mask_con_nans = df_limpio.isnull().any(axis=1)
mask_atipico_0 = df_limpio["Atipico"] == 0
mask_eliminar = mask_con_nans & mask_atipico_0

registros_a_eliminar = mask_eliminar.sum()

print(f"\n🔍 ANÁLISIS DE REGISTROS CON NaN:")
print(f"   Total registros con NaN: {mask_con_nans.sum():,}")
print(f"   Registros atípico=0 con NaN (a eliminar): {registros_a_eliminar:,}")

# Análisis detallado de registros atípicos con NaN
mask_atipico_1_con_nan = (df_limpio["Atipico"] == 1) & mask_con_nans
registros_atipicos_con_nan = mask_atipico_1_con_nan.sum()

print(f"   Registros atípico=1 con NaN (protegidos): {registros_atipicos_con_nan:,}")

# 3. ANÁLISIS DETALLADO DE NaN EN REGISTROS ATÍPICOS
if registros_atipicos_con_nan > 0:
    print(f"\n🔍 ANÁLISIS DETALLADO DE NaN EN REGISTROS ATÍPICOS (Atipico==1)")
    print("-" * 60)

    # Obtener solo registros atípicos
    df_atipicos = df_limpio[df_limpio["Atipico"] == 1].copy()

    # Contar NaN por columna en registros atípicos
    nan_por_columna_atipicos = df_atipicos.isnull().sum()
    nan_porcentaje_atipicos = (nan_por_columna_atipicos / len(df_atipicos)) * 100

    nan_analysis_atipicos = pd.DataFrame(
        {
            "Columna": nan_por_columna_atipicos.index,
            "NaN_Count": nan_por_columna_atipicos.values,
            "Porcentaje": nan_porcentaje_atipicos.values,
        }
    ).sort_values("NaN_Count", ascending=False)

    # Mostrar solo columnas con NaN
    nan_columns_atipicos = nan_analysis_atipicos[nan_analysis_atipicos["NaN_Count"] > 0]

    if len(nan_columns_atipicos) > 0:
        print(f"📊 Columnas con NaN en registros atípicos:")
        for _, row in nan_columns_atipicos.iterrows():
            print(
                f"   {row['Columna']}: {row['NaN_Count']:,} NaN ({row['Porcentaje']:.2f}%)"
            )
    else:
        print("✅ No hay NaN en registros atípicos")

# Eliminar registros (solo atípico == 0 con NaN)
df_final = df_limpio[~mask_eliminar]

# 4. RESUMEN FINAL
print(f"\n📊 ESTADO FINAL:")
print(f"   Total registros: {len(df_final):,}")

if len(df_final) > 0:
    atipicos_final = (df_final["Atipico"] == 1).sum()
    normales_final = (df_final["Atipico"] == 0).sum()

    print(f"   Registros atípicos (1): {atipicos_final:,}")
    print(f"   Registros normales (0): {normales_final:,}")
    print(f"   Total columnas: {df_final.shape[1]}")

    # Calcular porcentajes
    if len(df_final) > 0:
        porcentaje_atipicos = (atipicos_final / len(df_final)) * 100
        porcentaje_normales = (normales_final / len(df_final)) * 100

        print(f"\n📈 DISTRIBUCIÓN FINAL:")
        print(f"   Atípicos: {porcentaje_atipicos:.2f}%")
        print(f"   Normales: {porcentaje_normales:.2f}%")

    # Pérdida de datos
    perdida_registros = len(df_converted) - len(df_final)
    perdida_columnas = df_converted.shape[1] - df_final.shape[1]

    print(f"\n📉 RESUMEN DE ELIMINACIONES:")
    print(f"   Registros eliminados: {perdida_registros:,}")
    print(f"   Columnas eliminadas: {perdida_columnas}")
    print(
        f"   Porcentaje registros perdidos: {(perdida_registros / len(df_converted)) * 100:.2f}%"
    )

    # Verificar preservación de atípicos
    atipicos_inicial = (df_converted["Atipico"] == 1).sum()
    atipicos_preservados = atipicos_final

    print(f"\n🔒 VERIFICACIÓN ATÍPICOS:")
    print(f"   Atípicos iniciales: {atipicos_inicial:,}")
    print(f"   Atípicos finales: {atipicos_preservados:,}")

    if atipicos_inicial == atipicos_preservados:
        print("   ✅ TODOS los registros atípicos fueron preservados")
    else:
        atipicos_perdidos = atipicos_inicial - atipicos_preservados
        print(f"   ⚠️ Se perdieron {atipicos_perdidos:,} registros atípicos")

    # Verificar que no hay NaN en registros normales
    nans_en_normales = df_final[df_final["Atipico"] == 0].isnull().any().any()
    print(
        f"\n✅ Verificación NaN en registros normales: {'❌ AÚN HAY NaN' if nans_en_normales else '✅ SIN NaN'}"
    )

else:
    print("❌ No quedaron registros después del proceso de limpieza")

# Actualizar dataframe principal
print(f"\n✅ Proceso de limpieza completado")
print(f"📊 Dataset final: {df_final.shape[0]:,} filas × {df_final.shape[1]} columnas")

🔧 PROCESO REAL DE LIMPIEZA DE DATOS

1️⃣ ELIMINANDO COLUMNAS ESPECÍFICAS
----------------------------------------
📊 Columnas a eliminar encontradas: 8
   ✅ CreditoID
   ✅ PersonaID
   ✅ storeIdEventoA
   ✅ FechaEventoA
   ✅ EventoA
   ✅ FechaEventoB
   ✅ LocalizacionEventoB
   ✅ LocalizacionComercioCredito

📊 Columnas antes: 27
📊 Columnas después: 19
📊 Columnas eliminadas: 8

2️⃣ ELIMINANDO REGISTROS CON NaN (SOLO ATÍPICO == 0)
----------------------------------------
📊 ESTADO INICIAL:
   Total registros: 1,744,667
   Registros atípicos (1): 403
   Registros normales (0): 1,744,264

🔍 ANÁLISIS DE REGISTROS CON NaN:
   Total registros con NaN: 207,256
   Registros atípico=0 con NaN (a eliminar): 207,255
   Registros atípico=1 con NaN (protegidos): 1

🔍 ANÁLISIS DETALLADO DE NaN EN REGISTROS ATÍPICOS (Atipico==1)
------------------------------------------------------------
📊 Columnas con NaN en registros atípicos:
   StatusComercioCredito: 1 NaN (0.25%)

📊 ESTADO FINAL:
   Total registro

In [8]:
# 🔄 IMPUTACIÓN POR MODA
print("🔄 IMPUTACIÓN POR MODA")
print("=" * 40)

# Crear copia del dataset final
df_imputacion = df_final.copy()

# Imputar valores faltantes por moda
for col in df_imputacion.columns:
    if df_imputacion[col].isnull().sum() > 0:
        moda = df_imputacion[col].mode()[0]
        df_imputacion[col].fillna(moda, inplace=True)
        print(f"✅ {col}: Imputado por moda")

# Verificar resultado
valores_faltantes = df_imputacion.isnull().sum().sum()
print(f"\n📊 Valores faltantes: {valores_faltantes}")
print(f"📊 Dataset final: {df_imputacion.shape}")

if valores_faltantes == 0:
    print("✅ Dataset completamente limpio")
else:
    print("❌ Aún hay valores faltantes")

🔄 IMPUTACIÓN POR MODA
✅ StatusComercioCredito: Imputado por moda

📊 Valores faltantes: 0
📊 Dataset final: (1537412, 19)
✅ Dataset completamente limpio


/tmp/ipykernel_13304/2151223467.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_imputacion[col].fillna(moda, inplace=True)


In [9]:
# 🔍 ELIMINACIÓN DE DUPLICADOS
print("🔍 ELIMINACIÓN DE DUPLICADOS")
print("=" * 40)

duplicados_antes = df_imputacion.duplicated().sum()
print(f"📊 Duplicados encontrados: {duplicados_antes}")

if duplicados_antes > 0:
    df_imputacion = df_imputacion.drop_duplicates()
    print(f"✅ {duplicados_antes} duplicados eliminados")

print(f"📊 Registros finales: {len(df_imputacion):,}")


🔍 ELIMINACIÓN DE DUPLICADOS
📊 Duplicados encontrados: 0
📊 Registros finales: 1,537,412


**2. Selección de atributos (Feature Selection)**

In [10]:
# 🔧 GENERANDO VARIABLES DERIVADAS ESPECÍFICAS
print("🔧 GENERANDO VARIABLES DERIVADAS ESPECÍFICAS")
print("=" * 50)

# Crear copia del DataFrame actual
df_engineered = df_imputacion.copy()

try:
    # Verificar si FechaCredito existe y es datetime
    if "FechaCredito" in df_engineered.columns:
        # Variables temporales
        df_engineered["dia_credito"] = df_engineered["FechaCredito"].dt.day
        df_engineered["mes_credito"] = df_engineered["FechaCredito"].dt.month
        df_engineered["hora_credito"] = df_engineered["FechaCredito"].dt.hour
        df_engineered["dia_semana"] = df_engineered["FechaCredito"].dt.weekday
        print(
            "✅ Variables temporales: dia_credito, mes_credito, hora_credito, dia_semana"
        )

        # Es día de pago
        df_engineered["es_dia_pago"] = (
            (df_engineered["dia_credito"].isin([15, 30]))
            | (
                df_engineered["dia_credito"].isin([28, 29])
                & (df_engineered["mes_credito"] == 2)
            )
        ).astype(int)
        print("✅ es_dia_pago")

        # Es noche o madrugada
        df_engineered["es_noche_o_madrugada"] = (
            (df_engineered["hora_credito"] < 6) | (df_engineered["hora_credito"] >= 22)
        ).astype(int)
        print("✅ es_noche_o_madrugada")
    else:
        print("⚠️ FechaCredito no disponible - omitiendo variables temporales")

    # Frecuencia alta de créditos
    if "FrecuenciaCreditosSemana" in df_engineered.columns:
        df_engineered["frecuencia_categoria_alta"] = (
            df_engineered["FrecuenciaCreditosSemana"] > 3
        ).astype(int)
        print("✅ frecuencia_categoria_alta")

    # Match departamento
    if (
        "DepartamentoResidencia" in df_engineered.columns
        and "DepartamentoCredito" in df_engineered.columns
    ):
        df_engineered["departamento_match_residencia_credito"] = (
            df_engineered["DepartamentoResidencia"]
            == df_engineered["DepartamentoCredito"]
        ).astype(int)
        print("✅ departamento_match_residencia_credito")

    # Cupo alto utilizado
    if "ValorCredito" in df_engineered.columns and "CupoTotal" in df_engineered.columns:
        df_engineered["cupo_alto_utilizado"] = (
            (df_engineered["ValorCredito"] / df_engineered["CupoTotal"]) > 0.8
        ).astype(int)
        print("✅ cupo_alto_utilizado")

    print(f"\n📊 RESUMEN:")
    print(
        f"Dataset con variables derivadas: {df_engineered.shape[0]:,} filas × {df_engineered.shape[1]} columnas"
    )
    print("✅ Variables derivadas creadas exitosamente")

except Exception as e:
    print(f"❌ Error al crear variables derivadas: {e}")
    df_engineered


🔧 GENERANDO VARIABLES DERIVADAS ESPECÍFICAS
✅ Variables temporales: dia_credito, mes_credito, hora_credito, dia_semana
✅ es_dia_pago
✅ es_noche_o_madrugada
✅ frecuencia_categoria_alta
✅ departamento_match_residencia_credito
✅ cupo_alto_utilizado

📊 RESUMEN:
Dataset con variables derivadas: 1,537,412 filas × 28 columnas
✅ Variables derivadas creadas exitosamente


In [11]:
# 🗑️ ELIMINACIÓN DE COLUMNAS DATETIME
print("🗑️ ELIMINACIÓN DE COLUMNAS DATETIME")
print("=" * 40)

# Identificar columnas datetime
datetime_columns = df_engineered.select_dtypes(
    include=["datetime64[ns]", "datetime"]
).columns.tolist()

# Agregar FechaCredito específicamente si existe
if "FechaCredito" in df_engineered.columns:
    datetime_columns.append("FechaCredito")

# Eliminar duplicados
datetime_columns = list(set(datetime_columns))

print(f"📊 Columnas datetime encontradas: {len(datetime_columns)}")
for col in datetime_columns:
    print(f"   ❌ {col}")

if datetime_columns:
    # Eliminar columnas datetime
    df_engineered = df_engineered.drop(columns=datetime_columns)
    print(f"\n✅ {len(datetime_columns)} columnas datetime eliminadas")
else:
    print("\n⚠️ No se encontraron columnas datetime")

print(
    f"📊 Dataset final: {df_engineered.shape[0]:,} filas × {df_engineered.shape[1]} columnas"
)

🗑️ ELIMINACIÓN DE COLUMNAS DATETIME
📊 Columnas datetime encontradas: 1
   ❌ FechaCredito

✅ 1 columnas datetime eliminadas
📊 Dataset final: 1,537,412 filas × 27 columnas


In [12]:
df_engineered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1537412 entries, 0 to 1744665
Data columns (total 27 columns):
 #   Column                                 Non-Null Count    Dtype   
---  ------                                 --------------    -----   
 0   DepartamentoResidencia                 1537412 non-null  category
 1   DepartamentoMayorFrecuenciaCompra      1537412 non-null  category
 2   AlmacenMayorFrecuenciaPago             1537412 non-null  string  
 3   ValorPagosUltimosMes                   1537412 non-null  float64 
 4   AlmacenCredito                         1537412 non-null  string  
 5   RiesgoAlmacen                          1537412 non-null  category
 6   DepartamentoCredito                    1537412 non-null  category
 7   ValorCredito                           1537412 non-null  float64 
 8   CupoTotal                              1537412 non-null  float64 
 9   CupoDisponibleTotal                    1537412 non-null  float64 
 10  StatusComercioCredito              

In [13]:
# 🗑️ ELIMINACIÓN DE COLUMNAS STRING
print("🗑️ ELIMINACIÓN DE COLUMNAS STRING")
print("=" * 40)

# Identificar columnas string/object
string_columns = df_engineered.select_dtypes(
    include=["object", "string"]
).columns.tolist()

print(f"📊 Columnas string encontradas: {len(string_columns)}")
for col in string_columns:
    print(f"   ❌ {col}")

if string_columns:
    # Eliminar columnas string
    df_engineered = df_engineered.drop(columns=string_columns)
    print(f"\n✅ {len(string_columns)} columnas string eliminadas")
else:
    print("\n⚠️ No se encontraron columnas string")

print(
    f"📊 Dataset final: {df_engineered.shape[0]:,} filas × {df_engineered.shape[1]} columnas"
)


🗑️ ELIMINACIÓN DE COLUMNAS STRING
📊 Columnas string encontradas: 2
   ❌ AlmacenMayorFrecuenciaPago
   ❌ AlmacenCredito

✅ 2 columnas string eliminadas
📊 Dataset final: 1,537,412 filas × 25 columnas


**3. Ingeniería de atributos (Feature Engineering), escalado y encoding**

In [ ]:
# 📊 DISCRETIZACIÓN DE VARIABLES CONTINUAS PARA MODELOS CATEGÓRICOS
print("📊 DISCRETIZACIÓN DE VARIABLES CONTINUAS PARA MODELOS CATEGÓRICOS")
print("=" * 60)

# Crear copia del DataFrame
modelosvarcat = df_engineered.copy()

# Variables continuas a discretizar
variables_continuas = [
    "ValorPagosUltimosMes",
    "ValorCredito",
    "CupoTotal",
    "CupoDisponibleTotal",
    "ValorAtipicoCliente",
    "ValorAtipicoComercio",
]

print("🔧 Discretizando variables continuas:")
print("-" * 40)

for col in variables_continuas:
    if col in modelosvarcat.columns:
        try:
            # Primero intentar con qcut (quantiles)
            modelosvarcat[f"{col}_discretized"] = pd.qcut(
                modelosvarcat[col],
                q=5,  # 5 bins (Muy Bajo, Bajo, Medio, Alto, Muy Alto)
                labels=["Muy_Bajo", "Bajo", "Medio", "Alto", "Muy_Alto"],
                duplicates="drop",
            )
            print(f"   ✅ {col} → {col}_discretized (5 categorías con quantiles)")

        except ValueError as e:
            # Si qcut falla, usar cut con rangos uniformes
            try:
                modelosvarcat[f"{col}_discretized"] = pd.cut(
                    modelosvarcat[col],
                    bins=5,
                    labels=["Muy_Bajo", "Bajo", "Medio", "Alto", "Muy_Alto"],
                    include_lowest=True,
                )
                print(
                    f"   ✅ {col} → {col}_discretized (5 categorías con rangos uniformes)"
                )

            except Exception as e2:
                print(f"   ❌ Error discretizando {col}: {e2}")
                continue

        # Eliminar la variable continua original solo si se creó la discretizada
        if f"{col}_discretized" in modelosvarcat.columns:
            modelosvarcat = modelosvarcat.drop(columns=[col])

print(f"\n📊 Variables discretizadas creadas")
print(f"📊 Shape del dataset: {modelosvarcat.shape}")

# Crear carpeta Silver si no existe
silver_path = Path("../../data/Silver")
silver_path.mkdir(parents=True, exist_ok=True)

# 📊 ANÁLISIS DE CUANTILES PARA INTERPRETACIÓN DE DISCRETIZACIÓN
print("\n📊 ANÁLISIS DE CUANTILES PARA INTERPRETACIÓN DE DISCRETIZACIÓN")
print("=" * 70)

# Variables originales que fueron discretizadas
variables_continuas = [
    "ValorPagosUltimosMes",
    "ValorCredito",
    "CupoTotal",
    "CupoDisponibleTotal",
    "ValorAtipicoCliente",
    "ValorAtipicoComercio",
]

# DataFrame para almacenar los rangos de discretización
rangos_discretizacion = {}

print("🔍 ANÁLISIS DETALLADO POR VARIABLE:")
print("=" * 70)

for col in variables_continuas:
    if col in df_engineered.columns:
        print(f"\n📈 VARIABLE: {col}")
        print("-" * 50)

        # Calcular estadísticas básicas
        data = df_engineered[col]
        stats = {
            "count": data.count(),
            "mean": data.mean(),
            "std": data.std(),
            "min": data.min(),
            "max": data.max(),
        }

        print(f"📊 Estadísticas básicas:")
        print(f"   📝 Registros: {stats['count']:,}")
        print(f"   📈 Media: ${stats['mean']:,.2f}")
        print(f"   📊 Desv. Estándar: ${stats['std']:,.2f}")
        print(f"   📉 Mínimo: ${stats['min']:,.2f}")
        print(f"   📊 Máximo: ${stats['max']:,.2f}")

        # Calcular quintiles (5 grupos)
        quintiles = data.quantile([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])

        print(f"\n🎯 Quintiles (5 grupos):")
        labels_quintiles = ["Muy_Bajo", "Bajo", "Medio", "Alto", "Muy_Alto"]

        for i in range(5):
            inicio = quintiles.iloc[i]
            fin = quintiles.iloc[i + 1]
            porcentaje = 20  # Cada quintil representa 20%

            print(
                f"   {labels_quintiles[i]:>10}: ${inicio:>10,.0f} - ${fin:>10,.0f} ({porcentaje}% de datos)"
            )

        # Guardar rangos para interpretación posterior
        rangos_discretizacion[col] = {
            "quintiles": quintiles.to_dict(),
            "labels": labels_quintiles,
            "stats": stats,
        }

        # Si existe la variable discretizada, mostrar distribución
        col_discretized = f"{col}_discretized"
        if col_discretized in modelosvarcat.columns:
            print(f"\n📊 Distribución de la variable discretizada ({col_discretized}):")
            distribucion = modelosvarcat[col_discretized].value_counts().sort_index()

            for categoria in labels_quintiles:
                if categoria in distribucion.index:
                    count = distribucion[categoria]
                    percentage = (count / len(modelosvarcat)) * 100
                    print(
                        f"   {categoria:>10}: {count:>7,} registros ({percentage:>5.1f}%)"
                    )

print(f"\n💾 GUARDANDO INFORMACIÓN DE CUANTILES")
print("=" * 50)

# Crear DataFrame resumen de rangos
import json

# Crear resumen interpretable
resumen_rangos = []
for var, info in rangos_discretizacion.items():
    quintiles = info["quintiles"]
    labels = info["labels"]

    for i in range(5):
        inicio = quintiles[i / 5 if i == 0 else (i) / 5]
        fin = quintiles[(i + 1) / 5]

        resumen_rangos.append(
            {
                "Variable": var,
                "Categoria": labels[i],
                "Rango_Inicio": inicio,
                "Rango_Fin": fin,
                "Interpretacion": f"${inicio:,.0f} - ${fin:,.0f}",
            }
        )

# Convertir a DataFrame
df_rangos = pd.DataFrame(resumen_rangos)

# Guardar como CSV en lugar de parquet para evitar problemas con PyArrow
rangos_path = silver_path / "rangos_discretizacion.csv"
df_rangos.to_csv(rangos_path, index=False)

# También guardar como JSON para fácil lectura
rangos_json_path = silver_path / "rangos_discretizacion.json"
with open(rangos_json_path, "w") as f:
    json.dump(rangos_discretizacion, f, indent=2, default=str)

print(f"✅ Rangos guardados en: {rangos_path}")
print(f"✅ Información JSON guardada en: {rangos_json_path}")

# Mostrar resumen final
print(f"\n📋 RESUMEN PARA INTERPRETACIÓN DE MODELOS")
print("=" * 60)
print("🎯 Cada categoría representa aproximadamente el 20% de los datos")
print("📊 Los rangos están basados en quintiles de la distribución original")
print("💡 Usar esta información para interpretar coeficientes de modelos")

display(df_rangos.head(10))

# Guardar dataset como CSV en lugar de parquet
output_path = silver_path / "modelosvarcat.csv"
modelosvarcat.to_csv(output_path, index=False)

print(f"\n✅ Dataset discretizado guardado en: {output_path}")
print(
    f"📊 Dataset final para modelos categóricos: {modelosvarcat.shape[0]:,} filas × {modelosvarcat.shape[1]} columnas"
)

📊 DISCRETIZACIÓN DE VARIABLES CONTINUAS PARA MODELOS CATEGÓRICOS


🔧 Discretizando variables continuas:
----------------------------------------
   ✅ ValorPagosUltimosMes → ValorPagosUltimosMes_discretized (5 categorías con quantiles)
   ✅ ValorCredito → ValorCredito_discretized (5 categorías con quantiles)
   ✅ CupoTotal → CupoTotal_discretized (5 categorías con rangos uniformes)
   ✅ CupoDisponibleTotal → CupoDisponibleTotal_discretized (5 categorías con quantiles)
   ✅ ValorAtipicoCliente → ValorAtipicoCliente_discretized (5 categorías con quantiles)
   ✅ ValorAtipicoComercio → ValorAtipicoComercio_discretized (5 categorías con quantiles)

📊 Variables discretizadas creadas
📊 Shape del dataset: (1537412, 25)

📊 ANÁLISIS DE CUANTILES PARA INTERPRETACIÓN DE DISCRETIZACIÓN
🔍 ANÁLISIS DETALLADO POR VARIABLE:

📈 VARIABLE: ValorPagosUltimosMes
--------------------------------------------------
📊 Estadísticas básicas:
   📝 Registros: 1,537,412
   📈 Media: $470,584.43
   📊 Desv. Estándar: $405,779.99
   📉 Mínimo: $194.00
   📊 Máximo: $4,614,212.00

🎯 Quinti

,Variable,Categoria,Rango_Inicio,Rango_Fin,Interpretacion
0,ValorPagosUltimosMes,Muy_Bajo,194.0,147503.2,"$194 - $147,503"
1,ValorPagosUltimosMes,Bajo,147503.2,268620.0,"$147,503 - $268,620"
2,ValorPagosUltimosMes,Medio,268620.0,435872.0,"$268,620 - $435,872"
3,ValorPagosUltimosMes,Alto,435872.0,733408.0,"$435,872 - $733,408"
4,ValorPagosUltimosMes,Muy_Alto,733408.0,4614212.0,"$733,408 - $4,614,212"
5,ValorCredito,Muy_Bajo,10000.0,87100.0,"$10,000 - $87,100"
6,ValorCredito,Bajo,87100.0,130000.0,"$87,100 - $130,000"
7,ValorCredito,Medio,130000.0,186700.0,"$130,000 - $186,700"
8,ValorCredito,Alto,186700.0,280000.0,"$186,700 - $280,000"
9,ValorCredito,Muy_Alto,280000.0,4697187.0,"$280,000 - $4,697,187"



✅ Dataset discretizado guardado en: ../../data/Silver/modelosvarcat.csv
📊 Dataset final para modelos categóricos: 1,537,412 filas × 25 columnas
